In [1]:
print("hello")

hello


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

from IPython.display import Image, display

In [9]:
! pip install torch
! pip install transformers
! pip install dotenv
! pip install langgraph
! pip install matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 37.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 14.3 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.7 MB/s eta 0:00:00


In [12]:
from langgraph.graph import StateGraph, END, START
from src.Agent.states import DebateAgentState
from src.Agent.functions import should_continue

# 모델 로드 없이 그래프 구조만 생성
def stock_agent():
    workflow = StateGraph(DebateAgentState)

    # 노드 (실제 함수 대신 lambda로 대체 → 모델 로드 불필요)
    dummy = lambda state: state
    workflow.add_node("[Initial] Optimist",  dummy)
    workflow.add_node("[Initial] Pessimist", dummy)
    workflow.add_node("[Debate] Optimist",   dummy)
    workflow.add_node("[Debate] Pessimist",  dummy)
    workflow.add_node("Consensus Generator", dummy)
    workflow.add_node("Save Session", dummy)

    workflow.add_edge(START, "[Initial] Optimist")
    workflow.add_edge("[Initial] Optimist",  "[Initial] Pessimist")
    workflow.add_edge("[Initial] Pessimist", "[Debate] Optimist")
    workflow.add_edge("[Debate] Optimist",   "[Debate] Pessimist")
    workflow.add_conditional_edges(
        "[Debate] Pessimist",
        should_continue,
        {"optimist": "[Debate] Optimist", "summary": "Consensus Generator"}
    )
    workflow.add_edge("Consensus Generator", "Save Session")
    workflow.add_edge("Save Session", END)

    return workflow.compile()

graph = stock_agent()

In [13]:
# 인라인 표시
png_bytes = graph.get_graph().draw_mermaid_png()
display(Image(png_bytes))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
# PNG 파일로 저장
output_path = "stock_agent_workflow.png"
with open(output_path, "wb") as f:
    f.write(png_bytes)
print(f"저장 완료: {output_path}")

NameError: name 'png_bytes' is not defined

In [ ]:
def save_agent_diagram():
    """에이전트 워크플로우를 PNG 이미지로 저장"""
    
    agent = stock_agent()
    
    try:
        # PNG 이미지로 저장
        from IPython.display import Image, display
        
        # 이미지 생성 및 저장
        img_data = agent.get_graph().draw_mermaid_png()
        
        # 파일로 저장
        with open("stock_agent_workflow.png", "wb") as f:
            f.write(img_data)
        
        print("✅ 워크플로우 다이어그램이 'stock_agent_workflow.png'로 저장되었습니다!")
        
        # Jupyter에서 이미지 표시
        display(Image(img_data))
        
        return img_data
        
    except ImportError:
        print("❌ 이미지 생성을 위해 추가 패키지가 필요합니다:")
        print("pip install pygraphviz")
        return None
    except Exception as e:
        print(f"❌ 이미지 저장 오류: {str(e)}")
        return None

# 이미지 저장
save_agent_diagram()

❌ 이미지 저장 오류: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
